# 발표용 대표 계량기 PNG 저장

- 대표 계량기: `H2.Z66`
- PF 이상치 예시 계량기: `H2.ZE66`
- 데이터 기준: DB raw + weather join
- 저장 결과:
  - correlation: `H2.Z66`의 `P` 기준 1행 heatmap
  - STL: `H2.Z66 - P`
  - sliding window: `H2.Z66 - P`
  - sliding window: `H2.Z66 - PF`
  - sliding window: `H2.ZE66 - PF`
  - PF anomaly focus: `H2.ZE66 - PF`
  - PF anomaly emphasis: `H2.ZE66 - PF`
  - PF daily event view: `H2.ZE66` 일별 `max PF`
  - PF daily event view: `H2.ZE66` 일별 `|PF|>1` count


In [1]:
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd
from statsmodels.tsa.seasonal import STL

from scripts.pipeline.preprocess import fetch_joined_data

plt.rcParams["font.family"] = "NanumGothic"
plt.rcParams["axes.unicode_minus"] = False

TARGET_METER = "H2.Z66"
PF_ANOMALY_METER = "H2.ZE66"
PF_CANDIDATES = ["PF", "PF1", "PF2", "PF3"]
YEARS = [2018, 2019, 2020, 2021, 2022, 2023]

OUTPUT_DIR = Path("outputs/raw_eda/presentation/png")
REP_OUTPUT_DIR = OUTPUT_DIR / TARGET_METER
PF_OUTPUT_DIR = OUTPUT_DIR / PF_ANOMALY_METER
REP_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PF_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CORR_PATH = REP_OUTPUT_DIR / f"{TARGET_METER}_correlation_focus_P_row.png"
STL_PATH = REP_OUTPUT_DIR / f"{TARGET_METER}_stl_P.png"
SLIDING_P_PATH = REP_OUTPUT_DIR / f"{TARGET_METER}_sliding_P.png"
SLIDING_PF_PATH = REP_OUTPUT_DIR / f"{TARGET_METER}_sliding_PF.png"
PF_SLIDING_PATH = PF_OUTPUT_DIR / f"{PF_ANOMALY_METER}_sliding_PF.png"
PF_ANOMALY_PATH = PF_OUTPUT_DIR / f"{PF_ANOMALY_METER}_PF_anomaly_focus.png"
PF_ANOMALY_EMPHASIS_PATH = PF_OUTPUT_DIR / f"{PF_ANOMALY_METER}_PF_anomaly_emphasis.png"
PF_DAILY_MAX_PATH = PF_OUTPUT_DIR / f"{PF_ANOMALY_METER}_daily_max_PF.png"
PF_DAILY_COUNT_PATH = PF_OUTPUT_DIR / f"{PF_ANOMALY_METER}_daily_pf_outlier_count.png"


In [2]:
def load_meter_df(meter_urn: str) -> pd.DataFrame:
    df = fetch_joined_data(meter_urn).copy()
    df["ts"] = pd.to_datetime(df["ts"], utc=True, errors="coerce")
    for column in df.columns:
        if column not in {"ts", "meter_urn"}:
            df[column] = pd.to_numeric(df[column], errors="coerce")
    return df.sort_values("ts").reset_index(drop=True)


def resolve_pf_column(df: pd.DataFrame) -> str:
    for column in PF_CANDIDATES:
        if column in df.columns and df[column].notna().any():
            return column
    raise ValueError("PF 계열 컬럼이 없습니다.")


def resolve_corr_columns(df: pd.DataFrame) -> list[str]:
    resolved = []
    candidate_columns = [column for column in df.columns if column not in {"ts", "meter_urn"}]
    for column in candidate_columns:
        series = pd.to_numeric(df[column], errors="coerce")
        if int(series.notna().sum()) < 2:
            continue
        if int(series.nunique(dropna=True)) < 2:
            continue
        resolved.append(column)
    return resolved


def month_pattern_message(ts_df: pd.DataFrame, feature: str) -> str:
    monthly = ts_df.groupby(ts_df["ts"].dt.month)[feature].median().dropna()
    if monthly.empty:
        return "월별 median 계산 불가"
    peak_month = int(monthly.idxmax())
    trough_month = int(monthly.idxmin())
    return f"월별 median 기준 최고 {peak_month}월, 최저 {trough_month}월"


def save_correlation_png(df: pd.DataFrame) -> pd.DataFrame:
    corr_columns = resolve_corr_columns(df)
    corr_df = df[corr_columns].corr(numeric_only=True)
    p_row = corr_df.loc[["P"], :]
    ordered_cols = list(p_row.columns)

    fig_w = max(8, len(ordered_cols) * 1.6)
    fig, ax = plt.subplots(figsize=(fig_w, 2.8))
    im = ax.imshow(p_row.values, cmap="coolwarm", vmin=-1, vmax=1, aspect="auto")
    ax.set_xticks(range(len(ordered_cols)))
    ax.set_yticks([0])
    ax.set_xticklabels(ordered_cols, rotation=30, ha="right")
    ax.set_yticklabels(["P"])
    ax.set_title(f"{TARGET_METER} | P 대 모든 feature 상관관계", fontsize=14, fontweight="bold")

    for j, column in enumerate(ordered_cols):
        ax.text(j, 0, f"{p_row.iloc[0, j]:.2f}", ha="center", va="center", fontsize=10, color="black")

    cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.04)
    cbar.set_label("Pearson corr")
    focus = p_row.T.drop(index=["P"], errors="ignore").sort_values(by="P", key=lambda s: s.abs(), ascending=False)
    top_lines = [f"P~{idx}: {val:.2f}" for idx, val in focus["P"].head(3).items()]
    fig.text(0.02, 0.02, "설명 포인트 | " + " | ".join(top_lines), fontsize=10)
    fig.tight_layout(rect=[0, 0.12, 1, 1])
    fig.savefig(CORR_PATH, dpi=180, bbox_inches="tight")
    plt.close(fig)
    return corr_df


def save_stl_png(df: pd.DataFrame, feature: str) -> pd.DataFrame:
    working = df[["ts", feature]].copy().dropna(subset=["ts"])
    working[feature] = pd.to_numeric(working[feature], errors="coerce")
    working = working.dropna(subset=[feature]).drop_duplicates(subset=["ts"]).sort_values("ts")
    series = working.set_index("ts")[feature].interpolate(method="linear", limit=24)
    result = STL(series, period=24 * 7, seasonal=13).fit()

    detail = pd.DataFrame({"observed": result.observed, "trend": result.trend, "seasonal": result.seasonal, "residual": result.resid})
    note = month_pattern_message(working, feature)

    fig, axes = plt.subplots(4, 1, figsize=(16, 10), sharex=True)
    fig.suptitle(f"{TARGET_METER} {feature} STL 분해", fontsize=15, fontweight="bold", y=0.985)
    fig.text(0.5, 0.945, f"설명 포인트: {note}", ha="center", fontsize=11)

    axes[0].plot(detail.index, detail["observed"], color="steelblue", lw=0.8)
    axes[0].set_ylabel("Observed")
    axes[1].plot(detail.index, detail["trend"], color="darkorange", lw=0.8)
    axes[1].set_ylabel("Trend")
    axes[2].plot(detail.index, detail["seasonal"], color="seagreen", lw=0.8)
    axes[2].set_ylabel("Seasonal")
    axes[3].plot(detail.index, detail["residual"], color="gray", lw=0.8)
    axes[3].axhline(0.0, color="black", linestyle="--", lw=0.8)
    axes[3].set_ylabel("Residual")

    for ax in axes:
        ax.grid(alpha=0.25)
        ax.xaxis.set_major_locator(mdates.YearLocator())
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y", tz=detail.index.tz))

    fig.tight_layout(rect=[0, 0, 1, 0.90])
    fig.savefig(STL_PATH, dpi=180, bbox_inches="tight")
    plt.close(fig)
    return detail.reset_index(names="ts")


def save_sliding_overlay_png(df: pd.DataFrame, feature: str, output_path: Path, meter_label: str) -> pd.DataFrame:
    working = df[["ts", feature]].copy()
    working[feature] = pd.to_numeric(working[feature], errors="coerce")
    working = working.dropna(subset=["ts", feature]).sort_values("ts")
    working["year"] = working["ts"].dt.year
    working = working.loc[working["year"].isin(YEARS)].copy()
    monthly = working.groupby(["year", working["ts"].dt.month])[feature].median().reset_index(name="median")

    year_colors = {2018: "tab:blue", 2019: "tab:orange", 2020: "tab:green", 2021: "tab:red", 2022: "tab:purple", 2023: "tab:brown"}
    fig, ax = plt.subplots(figsize=(14, 6))
    for year in YEARS:
        year_df = monthly.loc[monthly["year"] == year]
        if year_df.empty:
            continue
        ax.plot(year_df["ts"], year_df["median"], marker="o", lw=1.8, alpha=0.9, color=year_colors[year], label=str(year))

    peak = monthly.sort_values("median", ascending=False).iloc[0]
    trough = monthly.sort_values("median", ascending=True).iloc[0]
    criterion = "기준: 2018~2023 각 연도의 월별 median을 1~12월 축에 overlay하여 계절성/연도간 패턴 유사성을 비교"
    ax.set_title(f"{meter_label} {feature} sliding window", fontsize=14, fontweight="bold")
    ax.set_xlabel("Month")
    ax.set_ylabel(feature)
    ax.set_xticks(range(1, 13))
    ax.grid(alpha=0.25)
    ax.legend(title="연도", ncol=3, frameon=False)
    fig.text(0.02, 0.02, criterion, fontsize=10)
    fig.text(0.02, 0.05, f"최대 월별 median: {int(peak['year'])}-{int(peak['ts']):02d}, 최소 월별 median: {int(trough['year'])}-{int(trough['ts']):02d}", fontsize=10)
    fig.tight_layout(rect=[0, 0.08, 1, 1])
    fig.savefig(output_path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    return monthly


def save_pf_anomaly_png(df: pd.DataFrame, feature: str) -> tuple[pd.DataFrame, pd.DataFrame, pd.Timestamp, pd.Timestamp, int]:
    working = df[["ts", feature]].copy()
    working[feature] = pd.to_numeric(working[feature], errors="coerce")
    working = working.dropna(subset=["ts", feature]).sort_values("ts")
    working["abs_pf"] = working[feature].abs()
    working["is_outlier"] = working["abs_pf"] > 1.0

    extreme_idx = working[feature].idxmax()
    center_ts = working.loc[extreme_idx, "ts"]
    zoom_start = center_ts - pd.Timedelta(days=5)
    zoom_end = center_ts + pd.Timedelta(days=5)
    zoom_df = working.loc[(working["ts"] >= zoom_start) & (working["ts"] <= zoom_end)].copy()
    outliers = zoom_df.loc[zoom_df["is_outlier"]].copy()

    fig, axes = plt.subplots(2, 1, figsize=(14, 9), height_ratios=[1.0, 1.2])
    fig.suptitle(f"{PF_ANOMALY_METER} PF 이상치 강조", fontsize=15, fontweight="bold", y=0.985)

    full_df = working.loc[working["ts"].dt.year.isin(YEARS)].copy()
    axes[0].plot(full_df["ts"], full_df[feature], color="gray", lw=0.7)
    axes[0].scatter(full_df.loc[full_df["is_outlier"], "ts"], full_df.loc[full_df["is_outlier"], feature], color="crimson", s=8, zorder=5)
    axes[0].axhline(1.0, color="crimson", linestyle="--", lw=1)
    axes[0].axhline(-1.0, color="crimson", linestyle="--", lw=1)
    axes[0].axvspan(zoom_start, zoom_end, color="gold", alpha=0.2)
    axes[0].set_title("전체 기간 PF 시계열 + 확대 구간", loc="left", fontsize=12, fontweight="bold")
    axes[0].set_ylabel(feature)
    axes[0].grid(alpha=0.25)

    axes[1].plot(zoom_df["ts"], zoom_df[feature], color="gray", lw=0.9)
    axes[1].scatter(outliers["ts"], outliers[feature], color="crimson", s=20, zorder=5, label="|PF| > 1")
    axes[1].axhline(1.0, color="crimson", linestyle="--", lw=1)
    axes[1].axhline(-1.0, color="crimson", linestyle="--", lw=1)
    axes[1].set_title(f"최대 PF 중심 확대 ({zoom_start.date()} ~ {zoom_end.date()})", loc="left", fontsize=12, fontweight="bold")
    axes[1].set_xlabel("Timestamp")
    axes[1].set_ylabel(feature)
    axes[1].grid(alpha=0.25)
    axes[1].legend(frameon=False)
    axes[1].xaxis.set_major_locator(mdates.DayLocator(interval=2))
    axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d", tz=zoom_df["ts"].dt.tz))

    max_row = working.loc[extreme_idx]
    axes[1].annotate(
        f"최대 PF {max_row[feature]:.2f}",
        xy=(max_row["ts"], max_row[feature]),
        xytext=(10, 20),
        textcoords="offset points",
        arrowprops={"arrowstyle": "->", "color": "black"},
        fontsize=10,
    )

    criterion = "기준: 이상치 한 점만 자르면 전후 정상 구간 대비가 사라지므로, PF 최대값 시점을 기준으로 ±5일 확대해 spike 전후 맥락을 함께 보여줌"
    fig.text(0.02, 0.02, criterion, fontsize=10)
    fig.text(0.02, 0.05, f"최대 PF 시점: {center_ts} | 전체 |PF|>1 건수: {int(working['is_outlier'].sum())}", fontsize=10)
    fig.tight_layout(rect=[0, 0.08, 1, 0.95])
    fig.savefig(PF_ANOMALY_PATH, dpi=180, bbox_inches="tight")
    plt.close(fig)
    return working, zoom_df, zoom_start, zoom_end, int(working['is_outlier'].sum())


def save_pf_anomaly_emphasis_png(df: pd.DataFrame, feature: str) -> tuple[pd.Timestamp, pd.Timestamp, float]:
    working = df[["ts", feature]].copy()
    working[feature] = pd.to_numeric(working[feature], errors="coerce")
    working = working.dropna(subset=["ts", feature]).sort_values("ts")
    working["is_outlier"] = working[feature].abs() > 1.0

    extreme_idx = working[feature].idxmax()
    center_ts = working.loc[extreme_idx, "ts"]
    peak_value = float(working.loc[extreme_idx, feature])
    zoom_start = center_ts - pd.Timedelta(days=2)
    zoom_end = center_ts + pd.Timedelta(days=2)
    zoom_df = working.loc[(working["ts"] >= zoom_start) & (working["ts"] <= zoom_end)].copy()
    outliers = zoom_df.loc[zoom_df["is_outlier"]].copy()

    fig, axes = plt.subplots(2, 1, figsize=(14, 9), height_ratios=[1.0, 1.1])
    fig.suptitle(f"{PF_ANOMALY_METER} PF 물리적 이상치 강조", fontsize=16, fontweight="bold", y=0.985)

    full_df = working.loc[working["ts"].dt.year.isin(YEARS)].copy()
    axes[0].axhspan(-1, 1, color="palegreen", alpha=0.25, label="정상 범위 [-1, 1]")
    axes[0].plot(full_df["ts"], full_df[feature], color="0.55", lw=0.8)
    axes[0].scatter(full_df.loc[full_df["is_outlier"], "ts"], full_df.loc[full_df["is_outlier"], feature], color="crimson", s=18, zorder=5, label="|PF| > 1")
    axes[0].axhline(1.0, color="crimson", linestyle="--", lw=1)
    axes[0].axhline(-1.0, color="crimson", linestyle="--", lw=1)
    axes[0].axvspan(zoom_start, zoom_end, color="gold", alpha=0.25)
    axes[0].set_title("전체 PF 시계열에서 정상 범위 이탈 강조", loc="left", fontsize=12, fontweight="bold")
    axes[0].set_ylabel(feature)
    axes[0].set_ylim(-1.5, max(12.5, peak_value + 0.5))
    axes[0].grid(alpha=0.25)
    axes[0].legend(frameon=False, loc="upper right")

    axes[1].axhspan(-1, 1, color="palegreen", alpha=0.25)
    axes[1].plot(zoom_df["ts"], zoom_df[feature], color="0.45", lw=1.0, marker="o", markersize=3)
    axes[1].scatter(outliers["ts"], outliers[feature], color="crimson", s=36, zorder=5, label="|PF| > 1")
    axes[1].axhline(1.0, color="crimson", linestyle="--", lw=1)
    axes[1].axhline(-1.0, color="crimson", linestyle="--", lw=1)
    axes[1].set_title(f"최대 PF 중심 확대 ({zoom_start.date()} ~ {zoom_end.date()})", loc="left", fontsize=12, fontweight="bold")
    axes[1].set_xlabel("Timestamp")
    axes[1].set_ylabel(feature)
    axes[1].set_ylim(-1.5, max(12.5, peak_value + 0.5))
    axes[1].grid(alpha=0.25)
    axes[1].legend(frameon=False, loc="upper right")
    axes[1].xaxis.set_major_locator(mdates.DayLocator(interval=1))
    axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d", tz=zoom_df["ts"].dt.tz))

    axes[1].annotate(
        f"PF = {peak_value:.2f}",
        xy=(center_ts, peak_value),
        xytext=(15, -10),
        textcoords="offset points",
        arrowprops={"arrowstyle": "->", "color": "black"},
        fontsize=11,
        fontweight="bold",
    )

    fig.text(0.02, 0.02, "설명 포인트 | 정상 PF 범위는 -1~1인데, 일부 시점에서 11.33까지 치솟아 물리적으로 불가능한 측정 이상으로 해석", fontsize=10)
    fig.tight_layout(rect=[0, 0.06, 1, 0.96])
    fig.savefig(PF_ANOMALY_EMPHASIS_PATH, dpi=180, bbox_inches="tight")
    plt.close(fig)
    return zoom_start, zoom_end, peak_value


def save_pf_daily_max_png(df: pd.DataFrame, feature: str) -> pd.DataFrame:
    working = df[["ts", feature]].copy()
    working[feature] = pd.to_numeric(working[feature], errors="coerce")
    working = working.dropna(subset=["ts", feature]).sort_values("ts")
    working["date"] = working["ts"].dt.floor("D")
    daily = working.groupby("date", as_index=False)[feature].max().rename(columns={feature: "daily_max_pf"})
    daily["is_outlier_day"] = daily["daily_max_pf"] > 1.0

    fig, ax = plt.subplots(figsize=(15, 5))
    ax.axhspan(-1, 1, color="palegreen", alpha=0.25, label="정상 범위 [-1, 1]")
    ax.plot(daily["date"], daily["daily_max_pf"], color="0.45", lw=0.9)
    ax.scatter(daily.loc[daily["is_outlier_day"], "date"], daily.loc[daily["is_outlier_day"], "daily_max_pf"], color="crimson", s=22, zorder=5, label="일별 max PF > 1")
    ax.axhline(1.0, color="crimson", linestyle="--", lw=1)
    ax.set_title(f"{PF_ANOMALY_METER} 일별 max PF", fontsize=14, fontweight="bold")
    ax.set_xlabel("Date")
    ax.set_ylabel("daily max PF")
    ax.grid(alpha=0.25)
    ax.legend(frameon=False, loc="upper right")
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    fig.text(0.02, 0.02, "설명 포인트 | 특정 시간대 반복 패턴이라기보다 특정 날짜에만 비정상 spike가 나타나는지 확인", fontsize=10)
    fig.tight_layout(rect=[0, 0.06, 1, 1])
    fig.savefig(PF_DAILY_MAX_PATH, dpi=180, bbox_inches="tight")
    plt.close(fig)
    return daily


def save_pf_daily_outlier_count_png(df: pd.DataFrame, feature: str) -> pd.DataFrame:
    working = df[["ts", feature]].copy()
    working[feature] = pd.to_numeric(working[feature], errors="coerce")
    working = working.dropna(subset=["ts", feature]).sort_values("ts")
    working["date"] = working["ts"].dt.floor("D")
    working["is_outlier"] = working[feature].abs() > 1.0
    daily = working.groupby("date", as_index=False)["is_outlier"].sum().rename(columns={"is_outlier": "outlier_count"})
    daily = daily.loc[daily["outlier_count"] > 0].copy()

    fig, ax = plt.subplots(figsize=(15, 4.5))
    ax.bar(daily["date"], daily["outlier_count"], color="crimson", width=1.5)
    ax.set_title(f"{PF_ANOMALY_METER} 일별 |PF|>1 발생 횟수", fontsize=14, fontweight="bold")
    ax.set_xlabel("Date")
    ax.set_ylabel("outlier count")
    ax.grid(alpha=0.25, axis="y")
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    fig.text(0.02, 0.02, "설명 포인트 | 이상치가 특정 날짜에만 몰려 있는지 확인", fontsize=10)
    fig.tight_layout(rect=[0, 0.06, 1, 1])
    fig.savefig(PF_DAILY_COUNT_PATH, dpi=180, bbox_inches="tight")
    plt.close(fig)
    return daily


In [3]:
rep_df = load_meter_df(TARGET_METER)
rep_pf_feature = resolve_pf_column(rep_df)
pf_df = load_meter_df(PF_ANOMALY_METER)
pf_feature = resolve_pf_column(pf_df)

corr_df = save_correlation_png(rep_df)
stl_detail = save_stl_png(rep_df, "P")
sliding_p_monthly = save_sliding_overlay_png(rep_df, "P", SLIDING_P_PATH, TARGET_METER)
sliding_pf_monthly = save_sliding_overlay_png(rep_df, rep_pf_feature, SLIDING_PF_PATH, TARGET_METER)
pf_sliding_monthly = save_sliding_overlay_png(pf_df, pf_feature, PF_SLIDING_PATH, PF_ANOMALY_METER)
pf_full_df, pf_zoom_df, pf_zoom_start, pf_zoom_end, pf_outlier_count = save_pf_anomaly_png(pf_df, pf_feature)
pf_emphasis_start, pf_emphasis_end, pf_peak_value = save_pf_anomaly_emphasis_png(pf_df, pf_feature)
pf_daily_max = save_pf_daily_max_png(pf_df, pf_feature)
pf_daily_count = save_pf_daily_outlier_count_png(pf_df, pf_feature)

print(f"대표 계량기: {TARGET_METER}")
print(f"대표 계량기 PF 컬럼: {rep_pf_feature}")
print(f"PF 이상치 예시 계량기: {PF_ANOMALY_METER}")
print(f"이상치 예시 PF 컬럼: {pf_feature}")
print(f"saved: {CORR_PATH}")
print(f"saved: {STL_PATH}")
print(f"saved: {SLIDING_P_PATH}")
print(f"saved: {SLIDING_PF_PATH}")
print(f"saved: {PF_SLIDING_PATH}")
print(f"saved: {PF_ANOMALY_PATH}")
print(f"saved: {PF_ANOMALY_EMPHASIS_PATH}")
print(f"saved: {PF_DAILY_MAX_PATH}")
print(f"saved: {PF_DAILY_COUNT_PATH}")
print()
print("[Sliding/PF 기준]")
print("P sliding: 2018~2023 각 연도의 월별 median을 overlay해서 계절성/연도간 패턴 비교")
print(f"PF sliding(대표): {TARGET_METER}의 {rep_pf_feature}를 같은 방식으로 연도별 overlay")
print(f"PF sliding(이상치 예시): {PF_ANOMALY_METER}의 {pf_feature}를 같은 방식으로 연도별 overlay")
print(f"PF anomaly focus: {PF_ANOMALY_METER} 전체 기간에서 PF 최대값 시점을 기준으로 ±5일 확대")
print(f"PF anomaly 확대 구간: {pf_zoom_start} ~ {pf_zoom_end}")
print(f"PF anomaly 전체 |{pf_feature}|>1 건수: {pf_outlier_count}")
print(f"PF anomaly emphasis 확대 구간: {pf_emphasis_start} ~ {pf_emphasis_end} | peak={pf_peak_value:.2f}")
print(f"PF daily max outlier days: {int((pf_daily_max['daily_max_pf'] > 1).sum())}")
print(f"PF daily count active days: {int(pf_daily_count.shape[0])}")
print()
display(corr_df.round(3))
display(sliding_p_monthly.head())
display(sliding_pf_monthly.head())
display(pf_sliding_monthly.head())
display(pf_daily_max.head())
display(pf_daily_count.head())
display(pf_zoom_df.head())


대표 계량기: H2.Z66
대표 계량기 PF 컬럼: PF
PF 이상치 예시 계량기: H2.ZE66
이상치 예시 PF 컬럼: PF
saved: outputs/raw_eda/presentation/png/H2.Z66/H2.Z66_correlation_focus_P_row.png
saved: outputs/raw_eda/presentation/png/H2.Z66/H2.Z66_stl_P.png
saved: outputs/raw_eda/presentation/png/H2.Z66/H2.Z66_sliding_P.png
saved: outputs/raw_eda/presentation/png/H2.Z66/H2.Z66_sliding_PF.png
saved: outputs/raw_eda/presentation/png/H2.ZE66/H2.ZE66_sliding_PF.png
saved: outputs/raw_eda/presentation/png/H2.ZE66/H2.ZE66_PF_anomaly_focus.png
saved: outputs/raw_eda/presentation/png/H2.ZE66/H2.ZE66_PF_anomaly_emphasis.png
saved: outputs/raw_eda/presentation/png/H2.ZE66/H2.ZE66_daily_max_PF.png
saved: outputs/raw_eda/presentation/png/H2.ZE66/H2.ZE66_daily_pf_outlier_count.png

[Sliding/PF 기준]
P sliding: 2018~2023 각 연도의 월별 median을 overlay해서 계절성/연도간 패턴 비교
PF sliding(대표): H2.Z66의 PF를 같은 방식으로 연도별 overlay
PF sliding(이상치 예시): H2.ZE66의 PF를 같은 방식으로 연도별 overlay
PF anomaly focus: H2.ZE66 전체 기간에서 PF 최대값 시점을 기준으로 ±5일 확대
PF anomaly 확대 구간: 2023-1

,P,W,PF,PF1,PF2,PF3,P1,P2,P3,I1,I2,I3,U1,U2,U3,Q,f,W_in,Ta,Igm
P,1.000,-0.018,0.769,0.242,0.986,0.983,0.996,0.996,0.985,0.988,0.989,0.985,-0.010,-0.015,-0.017,0.998,0.042,-0.018,0.088,0.052
W,-0.018,1.000,-0.127,-0.071,-0.073,-0.009,-0.013,-0.017,-0.005,-0.006,-0.006,-0.006,0.124,0.108,0.078,-0.020,0.647,1.000,0.040,-0.020
PF,0.769,-0.127,1.000,0.662,0.769,0.764,0.771,0.777,0.764,0.765,0.764,0.763,-0.157,-0.145,-0.148,0.763,-0.027,-0.127,0.105,0.117
PF1,0.242,-0.071,0.662,1.000,0.277,0.236,0.243,0.243,0.237,0.241,0.240,0.236,0.112,0.125,0.118,0.241,0.027,-0.071,0.078,0.002
PF2,0.986,-0.073,0.769,0.277,1.000,0.979,0.989,0.989,0.979,0.981,0.982,0.978,-0.035,-0.042,-0.039,0.984,-0.001,-0.073,0.059,0.029
PF3,0.983,-0.009,0.764,0.236,0.979,1.000,0.987,0.987,0.998,0.995,0.993,0.998,-0.025,-0.031,-0.031,0.981,0.045,-0.009,0.069,0.037
P1,0.996,-0.013,0.771,0.243,0.989,0.987,1.000,0.999,0.989,0.991,0.992,0.988,-0.008,-0.013,-0.017,0.994,0.047,-0.013,0.094,0.056
P2,0.996,-0.017,0.777,0.243,0.989,0.987,0.999,1.000,0.989,0.991,0.992,0.988,-0.015,-0.019,-0.022,0.993,0.042,-0.017,0.090,0.055
P3,0.985,-0.005,0.764,0.237,0.979,0.998,0.989,0.989,1.000,0.996,0.995,0.999,-0.014,-0.020,-0.020,0.982,0.059,-0.005,0.094,0.048
I1,0.988,-0.006,0.765,0.241,0.981,0.995,0.991,0.991,0.996,1.000,0.995,0.996,-0.011,-0.017,-0.019,0.986,0.052,-0.006,0.084,0.045


,year,ts,median
0,2018,1,4510.924250
1,2018,2,4566.382625
2,2018,3,27.020250
3,2018,4,27.240583
4,2018,5,10844.160667


,year,ts,median
0,2018,1,0.672646
1,2018,2,0.658687
2,2018,3,0.629150
3,2018,4,0.597475
4,2018,5,0.726821


,year,ts,median
0,2022,3,0.602312
1,2022,4,0.496668
2,2022,5,0.498352
3,2022,6,0.500312
4,2022,7,0.576104


,date,daily_max_pf,is_outlier_day
0,2022-03-24 00:00:00+00:00,0.627344,False
1,2022-03-25 00:00:00+00:00,0.631521,False
2,2022-03-26 00:00:00+00:00,0.616452,False
3,2022-03-27 00:00:00+00:00,0.620288,False
4,2022-03-28 00:00:00+00:00,8.099685,True


,date,outlier_count
4,2022-03-28 00:00:00+00:00,1
46,2022-05-09 00:00:00+00:00,1
86,2022-06-18 00:00:00+00:00,2
99,2022-07-01 00:00:00+00:00,1
104,2022-07-06 00:00:00+00:00,1


,ts,PF,abs_pf,is_outlier
13916,2023-10-23 10:00:00+00:00,0.618905,0.618905,False
13917,2023-10-23 11:00:00+00:00,0.604879,0.604879,False
13918,2023-10-23 12:00:00+00:00,0.612630,0.612630,False
13919,2023-10-23 13:00:00+00:00,0.620313,0.620313,False
13920,2023-10-23 14:00:00+00:00,0.606366,0.606366,False
